# W4 / W5 results reproduction - CALM-Editor (CISC7021 Group 26)

Two layers, deliberately kept apart:

**Part A - rebuild the reported numbers from the evidence already in the repository.**
Pure JSON reads over `FONG CHIN WAI/w4_model_level_improve/results/audit_*.json`,
`FONG CHIN WAI/w4_model_level_improve/eval_*/w3_calm_stream_*.json` and
`FONG CHIN WAI/w4_sequential_stream_and_fix/*.json`. This is what the W4/W5 tables and
figures are supposed to say. Seconds, no GPU.

**Part B - re-run the three pipelines on this laptop** (RTX 4070 Laptop 8 GB, `./.venv`).
Deliberately small budget (5 training steps, 16 edits) so the whole notebook stays under a
few minutes. Part B is a *plumbing check*: it proves the scripts still run end-to-end after
the `FONG CHIN WAI/` reorganisation. It does **not** reproduce the metrics - the numbers it
prints come from a 5-step checkpoint and carry no evidential weight. Only Part A carries the
reported values.

Everything is driven through `subprocess` against the project virtualenv, so this notebook
runs on any Python 3 kernel (torch / matplotlib are not needed inside the kernel).

In [1]:
import json, os, subprocess, sys, time
from pathlib import Path


def find_repo(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / ".git").exists():
            return cand
    raise RuntimeError("repository root (.git) not found above " + str(start))


REPO = find_repo(Path.cwd().resolve())
EE = REPO / "EasyEdit"
W4 = REPO / "FONG CHIN WAI" / "w4_model_level_improve"
SEQ = REPO / "FONG CHIN WAI" / "w4_sequential_stream_and_fix"
W5 = REPO / "FONG CHIN WAI" / "w5_results_and_figures"

PY = REPO / ".venv" / "Scripts" / "python.exe"
if not PY.exists():
    PY = Path(sys.executable)

print("REPO :", REPO)
print("PY   :", PY)
checks = [
    ("EasyEdit/data/counterfact.json", EE / "data" / "counterfact.json"),
    ("EasyEdit/hparams/CALM/gpt2-small.yaml", EE / "hparams" / "CALM" / "gpt2-small.yaml"),
    ("gpt2-small cache", EE / "hugging_cache" / "gpt2-small"),
    ("W4 audit JSONs (results/)", W4 / "results"),
    ("W4 eval_* streams", W4 / "eval_budget256"),
    ("W4 5K results", SEQ / "w4_calm_fix_5k_results.json"),
    ("W5 figures dir", W5),
]
for label, p in checks:
    print("%-42s %-8s %s" % (label, "OK" if p.exists() else "MISSING", p))


def run_py(args, tag="", tail=22, timeout=3600):
    """Run a repo script with the project virtualenv; stream the tail of its output."""
    cmd = [str(PY)] + [str(a) for a in args]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(REPO), capture_output=True, text=True,
                          timeout=timeout, errors="replace")
    dt = time.time() - t0
    lines = (proc.stdout or "").splitlines() + (proc.stderr or "").splitlines()
    print("[%s] exit=%d elapsed=%.1fs" % (tag, proc.returncode, dt))
    for ln in lines[-tail:]:
        print("    " + ln)
    return proc.returncode, dt, lines

REPO : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26
PY   : C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\.venv\Scripts\python.exe
EasyEdit/data/counterfact.json             OK       C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\EasyEdit\data\counterfact.json
EasyEdit/hparams/CALM/gpt2-small.yaml      OK       C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\EasyEdit\hparams\CALM\gpt2-small.yaml
gpt2-small cache                           OK       C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\EasyEdit\hugging_cache\gpt2-small
W4 audit JSONs (results/)                  OK       C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\results
W4 eval_* streams                          OK       C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\eval_budget256
W4 5K results                              OK       C:\Users\07viv\Desktop\CALM-Editor-CISC7021

## Part A.1 - W4 model-level audit: the 22.2% that is really 3.1%

`raw` counts every edit the streaming harness marks as `rewrite_ok`. The audit re-instruments
the same checkpoint and asks a stricter question: did a **WRITE** (hypernetwork parameter
update) actually change the model's answer? Edits routed to side memory or rejected score
`rewrite_ok` because the answer already matched, not because the writer worked. Those are the
pseudo hits.

In [2]:
AUDIT_ROWS = []
for f in sorted((W4 / "results").glob("audit_*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    s = d["summary"]
    w, r, j = s.get("WRITE", {}), s.get("ROUTE", {}), s.get("REJECT", {})
    total = w.get("n", 0) + r.get("n", 0) + j.get("n", 0) or 1
    raw_ok = s.get("model_level_total_ok", 0)
    AUDIT_ROWS.append({
        "variant": f.stem.replace("audit_", ""),
        "edits": total,
        "write_n": w.get("n", 0),
        "write_hits": w.get("rewrite_ok", 0),
        "write_rate": w.get("real_write_hit_rate", 0.0),
        "reject_n": j.get("n", 0),
        "pseudo": j.get("rewrite_ok", 0),
        "raw_ok": raw_ok,
        "raw_rate": raw_ok / total,
        "sec": s.get("total_seconds", 0.0),
    })

hdr = ("| variant | edits | WRITE n | REAL write hits | REAL hit rate | pseudo hits (REJECT) "
       "| raw model-level ok | raw rate | verdict |")
sep = "|---|---:|---:|---:|---:|---:|---:|---:|---|"
print(hdr)
print(sep.replace("---", ":--").replace("--:", "--:"))
for r in AUDIT_ROWS:
    verdict = "raw inflated by pseudo hits" if r["pseudo"] > 0 and r["write_rate"] < 0.05 else "consistent"
    print("| %s | %d | %d | %d | %.1f%% | %d/%d | %d | %.1f%% | %s |" % (
        r["variant"], r["edits"], r["write_n"], r["write_hits"], 100 * r["write_rate"],
        r["pseudo"], r["reject_n"], r["raw_ok"], 100 * r["raw_rate"], verdict))

base = next((r for r in AUDIT_ROWS if r["variant"] == "baseline"), AUDIT_ROWS[0])
print("")
print("Headline: raw %.1f%% (%d/%d edits 'rewrote something') -> real WRITE hits %.1f%% (%d/%d WRITEs)."
      % (100 * base["raw_rate"], base["raw_ok"], base["edits"],
         100 * base["write_rate"], base["write_hits"], base["write_n"]))
print("The %.1f-point gap is %d REJECT edits whose answer already matched the target." %
      (100 * (base["raw_rate"] - base["write_rate"]), base["pseudo"]))

| variant | edits | WRITE n | REAL write hits | REAL hit rate | pseudo hits (REJECT) | raw model-level ok | raw rate | verdict |
|:--|:--:|:--:|:--:|:--:|:--:|:--:|:--:|:--|
| baseline | 500 | 128 | 4 | 3.1% | 107/107 | 111 | 22.2% | raw inflated by pseudo hits |
| coh_l0.1 | 500 | 128 | 5 | 3.9% | 82/82 | 87 | 17.4% | raw inflated by pseudo hits |
| coh_v2_l0.3_c0.6 | 500 | 128 | 3 | 2.3% | 28/28 | 31 | 6.2% | raw inflated by pseudo hits |
| v2_seed101 | 500 | 128 | 2 | 1.6% | 40/40 | 42 | 8.4% | raw inflated by pseudo hits |

Headline: raw 22.2% (111/500 edits 'rewrote something') -> real WRITE hits 3.1% (4/128 WRITEs).
The 19.1-point gap is 107 REJECT edits whose answer already matched the target.


## Part A.2 - the 500-edit stream variants

Every `eval_*/` directory holds one `w3_calm_stream_*_trained.json` run record. `imm rewrite`
is the harness rate; `model-only` is the same rate restricted to edits the writer had to make.

In [3]:
def summarize_stream(name, path):
    d = json.loads(path.read_text(encoding="utf-8"))
    dc = d.get("decision_counts", {}) or {}
    tot = sum(dc.values()) or 1
    return {
        "variant": name,
        "edits": d.get("max_edits"),
        "WRITE": dc.get("WRITE", 0),
        "ROUTE": dc.get("ROUTE", 0),
        "REJECT": dc.get("REJECT", 0),
        "imm": d.get("immediate_rewrite_rate", 0.0),
        "model_only": d.get("immediate_model_rewrite_rate", 0.0),
        "retention": "%s/%s" % (d.get("retention_ok", 0), len(d.get("sample_retention") or [])),
        "sec": d.get("total_seconds", 0.0),
    }


STREAM_ROWS = []
for p in sorted(W4.glob("eval_*")):
    if p.is_dir():
        for jf in sorted(p.glob("w3_calm_stream_*_trained.json")):
            STREAM_ROWS.append(summarize_stream(p.name.replace("eval_", ""), jf))

print("| variant | edits | WRITE | ROUTE | REJECT | imm rewrite | model-only | retention | sec |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for r in STREAM_ROWS:
    print("| %s | %s | %d | %d | %d | %.1f%% | %.1f%% | %s | %.0f |" % (
        r["variant"], r["edits"], r["WRITE"], r["ROUTE"], r["REJECT"],
        100 * r["imm"], 100 * r["model_only"], r["retention"], r["sec"]))
print("\n%d runs. None of the W4 levers (seed, scale, budget, coherence, ns-proj, key-hidden) "
      "moves model-only retention off ~0: the writer, not the gate, is the bottleneck." % len(STREAM_ROWS))

| variant | edits | WRITE | ROUTE | REJECT | imm rewrite | model-only | retention | sec |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| budget256 | 500 | 256 | 133 | 111 | 23.2% | 23.2% | 2/6 | 211 |
| budget512 | 500 | 340 | 51 | 109 | 22.8% | 22.8% | 1/6 | 209 |
| budget64 | 500 | 64 | 329 | 107 | 22.2% | 22.2% | 2/6 | 209 |
| coh_l0.1_c0.6_b128 | 500 | 128 | 290 | 82 | 17.4% | 17.4% | 1/6 | 211 |
| coh_l0.3_c0.6_b128 | 500 | 128 | 290 | 82 | 17.4% | 17.4% | 1/6 | 215 |
| hidden768_w48s1000 | 500 | 128 | 316 | 56 | 11.6% | 11.6% | 0/6 | 219 |
| keyhidden | 500 | 423 | 75 | 2 | 0.8% | 0.8% | 0/6 | 224 |
| keyhidden_b128 | 500 | 128 | 370 | 2 | 0.8% | 0.8% | 0/6 | 896 |
| nskeep16 | 500 | 2 | 490 | 8 | 1.6% | 1.6% | 0/6 | 213 |
| nskeep2 | 500 | 2 | 490 | 8 | 1.6% | 1.6% | 0/6 | 213 |
| nskeep4 | 500 | 2 | 490 | 8 | 1.6% | 1.6% | 0/6 | 214 |
| rank8_w48s1000 | 500 | 128 | 316 | 56 | 11.4% | 11.4% | 0/6 | 222 |
| ret0.2_pollrand | 500 | 128 | 364 | 8 | 1.6% | 1.6% | 0/6 | 206 |
| scal

## Part A.3 - the 1K and 5K lifelong runs

In [4]:
LONG_ROWS = []
for name, path in [("w4_calm_fix_5k", SEQ / "w4_calm_fix_5k_results.json"),
                   ("w3_calm_stream_500_baseline", W4 / "results" / "w3_calm_stream_500_trained.json")]:
    if not path.exists():
        continue
    d = json.loads(path.read_text(encoding="utf-8"))
    if "decision_counts" in d:
        LONG_ROWS.append(summarize_stream(name, path))

print("| run | edits | WRITE | ROUTE | REJECT | imm rewrite | model-only | retention | sec |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|")
for r in LONG_ROWS:
    print("| %s | %s | %d | %d | %d | %.1f%% | %.1f%% | %s | %.0f |" % (
        r["variant"], r["edits"], r["WRITE"], r["ROUTE"], r["REJECT"],
        100 * r["imm"], 100 * r["model_only"], r["retention"], r["sec"]))

seq = json.loads((SEQ / "w4_sequential_5k_results.json").read_text(encoding="utf-8"))
print("\nW4 5K campaign: %s (%s)" % (seq.get("milestone"), seq.get("date")))
for run in seq.get("runs", []):
    if isinstance(run, dict):
        print("  -", json.dumps({k: run[k] for k in list(run)[:6]}, ensure_ascii=False)[:220])
    else:
        print("  -", str(run)[:220])

| run | edits | WRITE | ROUTE | REJECT | imm rewrite | model-only | retention | sec |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| w4_calm_fix_5k | 5000 | 128 | 4870 | 2 | 97.5% | 0.1% | 9/9 | 2334 |
| w3_calm_stream_500_baseline | 500 | 128 | 323 | 49 | 10.2% | 10.2% | 0/6 | 440 |

W4 5K campaign: W4 sequential / lifelong editing (5K+ edits) (2026-10-31)
  - trained_5k
  - baseline_5k


## Part A.4 - redraw the two W5 figures from the audit JSONs

The figures are written back to `FONG CHIN WAI/w5_results_and_figures/`, replacing the
hand-made versions with ones derived directly from `results/audit_*.json`.

In [5]:
PLOT = r"""
import json, sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

w4 = Path(sys.argv[1]); w5 = Path(sys.argv[2])
rows = []
for f in sorted((w4 / "results").glob("audit_*.json")):
    d = json.loads(f.read_text(encoding="utf-8")); s = d["summary"]
    w, r, j = s.get("WRITE", {}), s.get("ROUTE", {}), s.get("REJECT", {})
    total = (w.get("n", 0) + r.get("n", 0) + j.get("n", 0)) or 1
    rows.append((f.stem.replace("audit_", ""), 100 * s.get("model_level_total_ok", 0) / total,
                 100 * w.get("real_write_hit_rate", 0.0)))
rows = [r for r in rows if r[0] == "baseline"] or rows
labels = [r[0] for r in rows]
raw = [r[1] for r in rows]; real = [r[2] for r in rows]

fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=200)
ax.bar(labels, real, color="#c0392b")
for i, v in enumerate(real):
    ax.text(i, v + 0.15, "%.1f%%" % v, ha="center", fontsize=9)
ax.set_ylabel("REAL write hit rate (%)")
ax.set_title("Model-level write quality (audited, WRITE edits only)")
ax.set_ylim(0, max(real) * 2 + 1)
fig.tight_layout(); p1 = w5 / "w5_audited_real_write_hit.png"; fig.savefig(p1); plt.close(fig)

fig, ax = plt.subplots(figsize=(4.6, 3.6), dpi=200)
ax.bar(["raw\n(harness)", "audited\n(real WRITE)"], [raw[0], real[0]], color=["#95a5a6", "#c0392b"])
for i, v in enumerate([raw[0], real[0]]):
    ax.text(i, v + 0.4, "%.1f%%" % v, ha="center", fontsize=10)
ax.set_ylabel("% of edits"); ax.set_title("W4 model-level rewrite: raw vs audited")
ax.set_ylim(0, raw[0] * 1.35)
fig.tight_layout(); p2 = w5 / "w5_raw_vs_audited_real_write_hit.png"; fig.savefig(p2); plt.close(fig)
print(p1); print(p2)
"""

plot_path = Path(os.environ.get("TEMP", ".")) / "_w5_plot.py"
plot_path.write_text(PLOT, encoding="utf-8")
rc, dt, lines = run_py([plot_path, W4, W5], tag="figures", tail=6)
print("figures regenerated in %.1fs" % dt if rc == 0 else "figure step failed - see output above")

[figures] exit=0 elapsed=1.2s
    C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w5_results_and_figures\w5_audited_real_write_hit.png
    C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w5_results_and_figures\w5_raw_vs_audited_real_write_hit.png
figures regenerated in 1.2s


## Part B - re-run the pipelines on this laptop (plumbing only)

Three scripts, three small budgets. Expect roughly 40-60 s each once the model is warm.

1. `train_accum_warm.py` - 5 accumulation-aware training steps on 32 cases (warm-start 4, batch 2).
2. `audit_per_edit.py` - the audit over the 16 edits with the checkpoint that step 1 produced.
3. `w3_calm_stream.py` - the sequential stream over the same 16 edits.

What these prove: the scripts still find their own repo root after the reorganisation, still
locate `EasyEdit/data` and `EasyEdit/hparams`, and still talk to the checkpoint under
`FONG CHIN WAI/w4_model_level_improve/ckpt/`. What they do **not** prove: anything about the
reported quality - 5 steps is not a trained hypernetwork.

In [6]:
SMOKE_CKPT = "hypernet_smoke_notebook.pt"
TRAIN_ARGS = [
    "FONG CHIN WAI/w4_model_level_improve/train_accum_warm.py",
    "--train-steps", 5, "--max-train-cases", 32, "--batch-size", 2,
    "--warm-start", 4, "--sanity-accum-length", 2, "--device", "cuda:0",
    "--ckpt-name", SMOKE_CKPT,
]
rc_train, t_train, train_lines = run_py(TRAIN_ARGS, tag="B1 train", tail=16)
CKPT = W4 / "ckpt" / SMOKE_CKPT
print("\ncheckpoint:", CKPT, "exists=" + str(CKPT.exists()))

[B1 train] exit=0 elapsed=19.6s
        pos  2: nll 2.741 -> 2.765 (worse)
    [ACCUM-TRAIN] accum sanity: improved positions 1/2 pattern=[1, 0]
    [ACCUM-TRAIN] checkpoint saved: C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\ckpt\hypernet_smoke_notebook.pt
    [ACCUM-TRAIN] log written: C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\FONG CHIN WAI\w4_model_level_improve\logs\train_accum_warm.log
    [ACCUM-TRAIN] done
    C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\.venv\Lib\site-packages\timm\models\hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
      warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
    2026-11-05 21:54:26,748 - easyeditor.editors.editor - INFO - Instantiating model
    11/05/2026 21:54:26 - INFO - easyeditor.editors.editor -   Instantiating model
    `torch_dtype` is deprecated! Use `dt

In [7]:
SMOKE_AUDIT = Path(os.environ.get("TEMP", ".")) / "smoke_audit_notebook.json"
AUDIT_ARGS = [
    "EasyEdit/examples/audit_per_edit.py",
    "--ckpt", CKPT, "--out", SMOKE_AUDIT, "--max-edits", 16,
]
rc_audit, t_audit, audit_lines = run_py(AUDIT_ARGS, tag="B2 audit", tail=10)
if SMOKE_AUDIT.exists():
    a = json.loads(SMOKE_AUDIT.read_text(encoding="utf-8"))["summary"]
    print("\nWRITE:", a.get("WRITE"), " ROUTE:", a.get("ROUTE"), " REJECT:", a.get("REJECT"))

[B2 audit] exit=0 elapsed=24.8s
      warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
    2026-11-05 21:54:48,741 - easyeditor.editors.editor - INFO - Instantiating model
    11/05/2026 21:54:48 - INFO - easyeditor.editors.editor -   Instantiating model
    `torch_dtype` is deprecated! Use `dtype` instead!
    
    Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 148/148 [00:00<00:00, 8792.84it/s]
    2026-11-05 21:54:48,992 - easyeditor.editors.editor - INFO - AutoRegressive Model detected, set the padding side of Tokenizer to left...
    11/05/2026 21:54:48 - INFO - easyeditor.editors.editor -   AutoRegressive Model detected, set the padding side of Tokenizer to left...
    The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.

WRITE: {'n': 15, 'rewrit

In [8]:
SMOKE_DIR = Path(os.environ.get("TEMP", ".")) / "smoke_stream_notebook"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
STREAM_ARGS = [
    "EasyEdit/examples/w3_calm_stream.py",
    "--max-edits", 16, "--hypernet-ckpt", CKPT, "--output-dir", SMOKE_DIR,
]
rc_stream, t_stream, stream_lines = run_py(STREAM_ARGS, tag="B3 stream", tail=14)
if rc_stream == 0:
    js = sorted(SMOKE_DIR.glob("w3_calm_stream_*_trained.json"))[-1]
    s = json.loads(js.read_text(encoding="utf-8"))
    print("\ndecisions:", s.get("decision_counts"))
    print("avg sec/edit: %.3f" % s.get("avg_seconds_per_edit", 0.0))

[B3 stream] exit=0 elapsed=28.4s
    memory_stats={'n_shards': 64, 'max_per_shard': 200, 'total_stored': 1, 'avg_entries_per_shard': 0.02, 'min_entries': 0, 'max_entries': 1}
    retention_samples=8 retained=1
    JSON result written to: C:\Users\07viv\AppData\Local\Temp\smoke_stream_notebook\w3_calm_stream_16_trained.json
    C:\Users\07viv\Desktop\CALM-Editor-CISC7021-Task1-Group26\.venv\Lib\site-packages\timm\models\hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
      warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
    2026-11-05 21:55:16,436 - easyeditor.editors.editor - INFO - Instantiating model
    11/05/2026 21:55:16 - INFO - easyeditor.editors.editor -   Instantiating model
    `torch_dtype` is deprecated! Use `dtype` instead!
    
    Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 148/148 [00:00<00:00, 9682.08it/s]
    2026-11-05 21:55:16,663 - easyeditor.edito

## Part B.5 - verdict

In [9]:
print("| step | exit | elapsed (s) | what it proves |")
print("|---|---:|---:|---|")
print("| B1 train_accum_warm (5 steps) | %d | %.1f | repo-root discovery + EasyEdit data/hparams + checkpoint write |" % (rc_train, t_train))
print("| B2 audit_per_edit (16 edits) | %d | %.1f | checkpoint loads back, WRITE/ROUTE/REJECT re-instrumented |" % (rc_audit, t_audit))
print("| B3 w3_calm_stream (16 edits) | %d | %.1f | full streaming pipeline, JSON result written |" % (rc_stream, t_stream))
all_ok = (rc_train == 0 and rc_audit == 0 and rc_stream == 0)
print("")
print("Pipelines executable after the FONG CHIN WAI/ reorganisation:", "YES" if all_ok else "NO")
print("Reported W4/W5 metrics come from Part A only; Part B runs a 5-step checkpoint on 16 edits.")

| step | exit | elapsed (s) | what it proves |
|---|---:|---:|---|
| B1 train_accum_warm (5 steps) | 0 | 19.6 | repo-root discovery + EasyEdit data/hparams + checkpoint write |
| B2 audit_per_edit (16 edits) | 0 | 24.8 | checkpoint loads back, WRITE/ROUTE/REJECT re-instrumented |
| B3 w3_calm_stream (16 edits) | 0 | 28.4 | full streaming pipeline, JSON result written |

Pipelines executable after the FONG CHIN WAI/ reorganisation: YES
Reported W4/W5 metrics come from Part A only; Part B runs a 5-step checkpoint on 16 edits.
